In [ ]:
!pip install transformers torch spacy scikit-learn jsonlines
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json
import jsonlines
import torch
import spacy
import numpy as np

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score

# -------- MODEL --------
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

nlp = spacy.load("en_core_web_sm")

# -------- LABELS --------
labels = {
    "TOPIC": "main subject or research topic",
    "METHOD": "method or technique used to perform a task",
    "CONCEPT": "general concept or idea in computer science",
    "ALGORITHM": "step by step computational algorithm",
    "OTHER": "other technical term"
}

# -------- EMBEDDING FUNCTION --------
def embed(texts):
    # Accepts a single string or list of strings
    if isinstance(texts, str):
        texts = [texts]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    # RoBERTa does not use token_type_ids
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")

    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling over tokens
    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()

# -------- LABEL EMBEDDINGS --------
label_embeddings = {label: embed(desc) for label, desc in labels.items()}

# -------- CANDIDATE EXTRACTION --------
def extract_candidates(text):
    doc = nlp(text)
    candidates = set()
    for chunk in doc.noun_chunks:
        candidates.add(chunk.text.strip())
    return list(candidates)

# -------- PREDICT LABEL BY COSINE SIMILARITY --------
def predict_entity_label(entity):
    e_vec = embed(entity)
    best_label = None
    best_score = -1

    for label, l_emb in label_embeddings.items():
        score = cosine_similarity(e_vec, l_emb)[0][0]
        if score > best_score:
            best_score = score
            best_label = label
    return best_label

# -------- LOAD DATA --------
data = []
with jsonlines.open("cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

# -------- 3-SHOT EXAMPLES --------
few_shot_examples = data[:3]  # first 3 samples
few_shot_entities = []

for sample in few_shot_examples:
    gold = json.loads(sample["output"])
    for g in gold:
        few_shot_entities.append({
            "entity": g["entity"].lower(),
            "label": g["label"]
        })

# -------- 3-SHOT PREDICTION --------
def predict_entities_3shot(text):
    candidates = extract_candidates(text)
    predictions = []

    for c in candidates:
        c_lower = c.lower()
        assigned_label = None

        # check against 3-shot examples
        for shot in few_shot_entities:
            if c_lower in shot["entity"] or shot["entity"] in c_lower:
                assigned_label = shot["label"]
                break

        # fallback to RoBERTa similarity
        if assigned_label is None:
            assigned_label = predict_entity_label(c)

        predictions.append({
            "entity": c,
            "label": assigned_label
        })

    return predictions

# -------- METRICS --------
def compute_accuracy(y_true, y_pred):
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

def compute_partial_accuracy(gold_entities, pred_entities):
    match = 0
    for g in gold_entities:
        g_text = g["entity"].lower()
        for p in pred_entities:
            p_text = p["entity"].lower()
            if g_text in p_text or p_text in g_text:
                match += 1
                break
    return match / len(gold_entities)

# -------- EVALUATION --------
y_true, y_pred, partial_scores = [], [], []

for sample in data:
    text = sample["input"]
    gold = json.loads(sample["output"])
    pred = predict_entities_3shot(text)

    gold_dict = {g["entity"]: g["label"] for g in gold}
    pred_dict = {p["entity"]: p["label"] for p in pred}

    for ent in gold_dict:
        y_true.append(gold_dict[ent])
        y_pred.append(pred_dict.get(ent, "OTHER"))

    partial_scores.append(compute_partial_accuracy(gold, pred))

# -------- REPORT --------
micro_precision = precision_score(y_true, y_pred, average="micro")
micro_recall = recall_score(y_true, y_pred, average="micro")
micro_f1 = f1_score(y_true, y_pred, average="micro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")
macro_f1 = f1_score(y_true, y_pred, average="macro")

accuracy = compute_accuracy(y_true, y_pred)
partial_accuracy = sum(partial_scores) / len(partial_scores)

print("\n===== Evaluation Results =====")
print("Accuracy:", round(accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))
print("\nMicro Precision:", round(micro_precision,4))
print("Micro Recall:", round(micro_recall,4))
print("Micro F1:", round(micro_f1,4))
print("\nMacro Precision:", round(macro_precision,4))
print("Macro Recall:", round(macro_recall,4))
print("Macro F1:", round(macro_f1,4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== Evaluation Results =====
Accuracy: 0.2418
Partial Accuracy: 0.9297

Micro Precision: 0.2418
Micro Recall: 0.2418
Micro F1: 0.2418

Macro Precision: 0.2074
Macro Recall: 0.1867
Macro F1: 0.066


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
